# LSTM vs Transformer — Model Comparison

Trains both models on identical data, then compares them using bootstrap
confidence intervals so the performance difference is statistically meaningful.

**Why bootstrapping?**  Test sets per stock are ~50–80 windows.  A single AUC
score from 60 samples is noisy — bootstrapping shows whether the gap between
models is real or sampling variance.  Overlapping 95% CIs = not significant.

In [ ]:
from sentiment.log import setup_logging
setup_logging()

In [ ]:
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 1. Load data

In [ ]:
import pandas as pd

from sentiment.sources.cache import MarketDataCache
from sentiment.sources.fundamental import FundamentalCache
from sentiment.sources.news.repository import ArticleRepository
from sentiment.features.technical import TechnicalFactors
from sentiment.features.dataloader import build_dataset, make_loaders

TICKER = "AAPL"
YEAR   = 2025
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {DEVICE}")


In [ ]:
# Market data
cache = MarketDataCache()
df = cache.load(TICKER, YEAR)
if df is None or df.empty:
    raise RuntimeError(f"No market data for {TICKER} {YEAR} — run market.ipynb first")

# Fundamental data — use load_df() which returns a DatetimeIndex'd DataFrame
# suitable for forward-filling inside build_dataset. Returns None if not cached.
fund_cache = FundamentalCache()
fund_df = fund_cache.load_df(TICKER)

print(f"market rows : {len(df)}")
print(f"fund rows   : {len(fund_df) if fund_df is not None else 'no data'}")

## 1b. Load & encode news sentiment

Articles are loaded from `ArticleRepository`, encoded via BART → FinBERT,
then cached as parquet so re-running skips the GPU-expensive step.
Set `USE_CACHED_SENTIMENT = False` to force re-encoding.

In [ ]:
# Load articles for this ticker from the news repository
_repo = ArticleRepository()
_articles: list = []
for _month in range(1, 13):
    _mdf = _repo.read_month(YEAR, _month)
    if _mdf.empty:
        continue
    _mask = _mdf["tickers"].apply(lambda t: TICKER in t)
    for _, _row in _mdf[_mask].iterrows():
        _articles.append({
            "id":           _row["id"],
            "url":          _row["url"],
            "title":        _row["title"] or "",
            "text":         _row["text"] or "",
            "publish_date": _row["publish_date"],
            "source_name":  _row["source_name"],
            "language":     _row["language"],
            "tickers":      _row["tickers"],
        })
print(f"articles found : {len(_articles)} for {TICKER} {YEAR}")


In [ ]:
from pathlib import Path

_SENT_CACHE = Path("data/sentiment") / f"{TICKER}_{YEAR}.parquet"
USE_CACHED_SENTIMENT = True   # set False to force re-encoding

if USE_CACHED_SENTIMENT and _SENT_CACHE.exists():
    sentiment_df = pd.read_parquet(_SENT_CACHE)
    print(f"loaded cached sentiment — {len(sentiment_df)} ticker-days")
elif _articles:
    from sentiment.embeddings.pipeline import SentimentPipeline
    _pipeline = SentimentPipeline(device=DEVICE)
    sentiment_df = _pipeline.process_ticker_articles({TICKER: _articles})
    _SENT_CACHE.parent.mkdir(parents=True, exist_ok=True)
    sentiment_df.to_parquet(_SENT_CACHE, index=False)
    print(f"encoded and cached sentiment — {len(sentiment_df)} ticker-days")
else:
    sentiment_df = None
    print("no articles found — training without sentiment")


In [ ]:
# Build fused dataset
technical = TechnicalFactors()
dataset = build_dataset(
    df,
    technical,
    sentiment_df=sentiment_df,
    ticker=TICKER,
    fundamental_df=fund_df,
)

N_FUND      = dataset["X_fundamental"].shape[1]
N_SENT_PROBS = dataset["X_sentiment_probs"].shape[-1]  # 3 with sentiment, 0 without
print(f"windows      : {len(dataset['y'])}")
print(f"n_fund       : {N_FUND}")
print(f"n_sent_probs : {N_SENT_PROBS}  (0 = sentiment disabled)")


In [ ]:
# Class balance — compute pos_weight for BCEWithLogitsLoss
from sentiment.features.dataloader import make_loaders as _ml
import inspect
_N = len(dataset["y"])
_test_start = int(_N * 0.8)
_val_start  = int(_test_start * 0.9)
y_train = dataset["y"][:_val_start]
n_pos   = int(y_train.sum())
n_neg   = len(y_train) - n_pos
pos_weight = n_neg / n_pos if n_pos > 0 else 1.0

print(f"Train class balance — positive: {n_pos} ({n_pos/len(y_train):.1%})  "
      f"negative: {n_neg} ({n_neg/len(y_train):.1%})")
print(f"pos_weight : {pos_weight:.3f}  (1.0 = balanced)")

In [ ]:
train_loader, val_loader, test_loader, tech_scaler, fund_scaler = make_loaders(
    dataset, batch_size=32
)

## 2. Train LSTM

In [ ]:
from sentiment.model import SentimentLSTM, train_model

lstm = SentimentLSTM(n_fundamentals=N_FUND, n_sentiment_probs=N_SENT_PROBS)

lstm_history = train_model(
    lstm,
    train_loader,
    val_loader,
    n_epochs=100,
    lr=1e-3,
    patience=15,
    device=DEVICE,
    seed=SEED,
    pos_weight=pos_weight,
)
print(f"LSTM best epoch {lstm_history['best_epoch']}  val_auc={lstm_history['best_val_auc']:.4f}")


## 3. Train Transformer (6-layer, paper default)

In [ ]:
from sentiment.model import SentimentTransformer

# Uses n_layers=6 by default (reference paper architecture)
transformer = SentimentTransformer(
    n_fundamentals=N_FUND,
    n_sentiment_probs=N_SENT_PROBS,
    d_model=64,
    nhead=4,
    dim_feedforward=128,
    dropout=0.2,
)

transformer_history = train_model(
    transformer,
    train_loader,
    val_loader,
    n_epochs=100,
    lr=1e-3,
    patience=15,
    device=DEVICE,
    seed=SEED,
    pos_weight=pos_weight,
)
print(f"Transformer best epoch {transformer_history['best_epoch']}  val_auc={transformer_history['best_val_auc']:.4f}")


## 4. Point estimates on test set

In [ ]:
from sentiment.model import evaluate

lstm_test        = evaluate(lstm,        test_loader, device=DEVICE)
transformer_test = evaluate(transformer, test_loader, device=DEVICE)

point_df = pd.DataFrame(
    {"LSTM": lstm_test, "Transformer": transformer_test},
    index=["auc", "accuracy", "precision", "recall", "loss"],
).round(4)
display(point_df)

## 5. Bootstrap confidence intervals

1 000 resamples with replacement.  If the 95% CIs for AUC **overlap**, the
performance difference is not statistically significant at the 5% level.

In [ ]:
from sentiment.model import bootstrap_evaluate

lstm_bs        = bootstrap_evaluate(lstm,        test_loader, device=DEVICE, n_bootstrap=1000, seed=SEED)
transformer_bs = bootstrap_evaluate(transformer, test_loader, device=DEVICE, n_bootstrap=1000, seed=SEED)

print(f"Bootstrap resamples — LSTM: {lstm_bs['n_bootstrap']}, Transformer: {transformer_bs['n_bootstrap']}")
print(f"Test samples: {lstm_bs['n_samples']}")

In [ ]:
def _fmt(result, metric):
    mean = result[f"{metric}_mean"]
    lo   = result[f"{metric}_ci_low"]
    hi   = result[f"{metric}_ci_high"]
    return f"{mean:.4f}  [{lo:.4f} – {hi:.4f}]"

metrics = ["auc", "accuracy", "precision", "recall"]
rows = {
    m: {"LSTM": _fmt(lstm_bs, m), "Transformer": _fmt(transformer_bs, m)}
    for m in metrics
}
bootstrap_df = pd.DataFrame(rows).T
bootstrap_df.index.name = "metric"
print("95% bootstrap confidence intervals")
display(bootstrap_df)

## 6. Training curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, metric, label in [
    (axes[0], "val_auc",  "Validation AUC"),
    (axes[1], "val_loss", "Validation Loss"),
]:
    ax.plot(lstm_history[metric],        label="LSTM",        linewidth=2)
    ax.plot(transformer_history[metric], label="Transformer", linewidth=2, linestyle="--")
    ax.axvline(lstm_history["best_epoch"] - 1,        color="C0", alpha=0.3, linestyle=":")
    ax.axvline(transformer_history["best_epoch"] - 1, color="C1", alpha=0.3, linestyle=":")
    ax.set_title(label)
    ax.set_xlabel("Epoch")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()